In [ ]:
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4216)]
        )
    except RuntimeError as e:
        print(e)

In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

num_classes = 7 

base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(num_classes, activation='softmax', dtype = 'float32')(x) 

model = Model(inputs=base_model.input, outputs=predictions)
criterion = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False) 

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-4
)

model.compile(optimizer=optimizer, loss=criterion, metrics=['accuracy'])
model.summary()

In [ ]:
import tensorflow as tf

def scale_and_normalize_keras(image_tensor):

    scaled_image = image_tensor / 255.0
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float16)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float16)
    mean = tf.reshape(mean, [1, 1, 3])
    std = tf.reshape(std, [1, 1, 3])
    normalized_image = (scaled_image - mean) / std
    return normalized_image

normalization_layer_keras = tf.keras.layers.Lambda(scale_and_normalize_keras, name='scale_and_normalize_layer')

In [ ]:
import tensorflow as tf

def reverse_scale_and_normalize_keras(normalized_image_tensor):
   
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float16)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float16)
    mean = tf.reshape(mean, [1, 1, 3])
    std = tf.reshape(std, [1, 1, 3])
    scaled_0_1_image = normalized_image_tensor * std + mean
    original_0_255_image = scaled_0_1_image * 255.0
    return tf.clip_by_value(original_0_255_image, 0.0, 255.0)


reverse_normalization_layer_keras = tf.keras.layers.Lambda(reverse_scale_and_normalize_keras, name='reverse_scale_and_normalize_layer'
)


In [ ]:
train_loader = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/train",
    image_size=(224, 224),
    batch_size=32
)

val_loader = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/val",
    image_size=(224, 224),
    batch_size=32
)

test_loader = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/test",
    image_size=(224, 224),
    batch_size=32
)

class_names = train_loader.class_names

train_ds = train_loader.map(lambda x, y: (normalization_layer_keras(x), y))
val_ds = val_loader.map(lambda x, y: (normalization_layer_keras(x), y))
test_ds = test_loader.map(lambda x, y: (normalization_layer_keras(x), y))

In [ ]:
EPOCHS = 50
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds, 
)

for epoch in range(EPOCHS):
    tr_loss = history.history['loss'][epoch]
    tr_acc = history.history['accuracy'][epoch]
    val_loss = history.history['val_loss'][epoch]
    val_acc = history.history['val_accuracy'][epoch]
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Acc: {tr_acc:.4f} : Train Loss: {tr_loss:.4f} | Val Acc: {val_acc:.4f} : Val Loss: {val_loss:.4f}")



In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models
from thop import profile
import psutil, time

model.save('MobileNetV1.keras')
model_size = os.path.getsize('MobileNetV1.keras') / (1024 ** 2)
print(f"Model File Size: {model_size:.2f} MB")
params = model.count_params()
print(f"Total Parameters: {params:,}")

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'b-', label='Training Accuracy')
plt.plot(epochs, loss, 'r-', label='Training Loss')
plt.title('Training:(Accuracy vs Loss)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.savefig("Training Accuracy Vs Loss MV1.png", dpi=300, bbox_inches="tight")

plt.subplot(1, 2, 2)
plt.plot(epochs, val_acc, 'b-', label='Validation Accuracy')
plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
plt.title('Validation:(Accuracy vs Loss)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig("Validation Accurscy Vs Loss MV1.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import confusion_matrix , classification_report

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

n_classes=cm.shape[0]

TP = np.zeros(n_classes, dtype=int)
FP = np.zeros(n_classes, dtype=int)
FN = np.zeros(n_classes, dtype=int)
TN = np.zeros(n_classes, dtype=int)

for i in range(n_classes):
    TP[i] = cm[i, i]
    FP[i] = cm[:, i].sum() - TP[i]
    FN[i] = cm[i, :].sum() - TP[i]
    TN[i] = cm.sum() - (TP[i] + FP[i] + FN[i])


plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.savefig("ConfusionMatrixMV1.png", dpi=300, bbox_inches="tight")
plt.show()


for i in range(n_classes):
    print(f"Class {class_names[i]}:>>>>>>>> TP={TP[i]}, FP={FP[i]}, FN={FN[i]}, TN={TN[i]}")

print("\nClassification Report:\n")
report = classification_report(y_true,y_pred,target_names=class_names)
print(report)
with open("classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

max_images_to_display = 10
images_displayed_count = 0

plt.figure(figsize=(20, math.ceil(max_images_to_display/5) * 4))

for image_batch, label_batch in test_ds:
    predictions_batch = model.predict(image_batch)

    for i in range(image_batch.shape[0]):
        if images_displayed_count >= max_images_to_display:
            break

        image = image_batch[i]
        true_label_idx = label_batch[i].numpy()
        predicted_label_idx = np.argmax(predictions_batch[i])
        denormalized_image = reverse_scale_and_normalize_keras(image)
        display_image = (denormalized_image.numpy()).astype(np.uint8)
        plt.subplot(math.ceil(max_images_to_display/5), 5, images_displayed_count + 1)
        plt.imshow(display_image)
        plt.title(f"True: {class_names[true_label_idx]}\nPred: {class_names[predicted_label_idx]}")
        plt.axis("off")
        images_displayed_count += 1
    
    if images_displayed_count >= max_images_to_display:
        break

plt.tight_layout()
plt.savefig("prediction from Test dataset MV1.png", dpi=300, bbox_inches="tight")
plt.show()